In [26]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import product

In [27]:
filename = r'Y:\Members\Mia-Sanjana-Hadent\Processed Data\FOR SANJANA Testing_Mouse1\combined_matrix_final_total1.csv'
combined_matrix = pd.read_csv( filename )

In [ ]:
combined_matrix

In [28]:
feature_cols = [col for col in combined_matrix.columns if col.startswith("Feature_")]

cluster_means = (
    combined_matrix.groupby("ClusterNumber")[feature_cols]
      .mean()
      .reset_index()
)

In [ ]:
# bins

# 1-11
# 12-22
# 23-28
# 29-30

In [29]:
df = cluster_means

In [ ]:
# option 1

# Suppose your feature dataframe is called df, with columns Feature_1 ... Feature_30
feature_cols = [col for col in df.columns if col.startswith("Feature_")]
totalMatrix = df[feature_cols].to_numpy()  # shape: (N, 30)

# Define bin (histogram) boundaries (Python uses 0-based indices)
bins = [(0, 11), (11, 22), (22, 28), (28, 30)]

# Initialize similarity matrix
N = totalMatrix.shape[0]
Dsim = np.zeros((N, N))

# Compute pairwise histogram intersection similarity
for i in range(N):
    # Get one sample
    vec_i = totalMatrix[i, :]
    
    # Compare it to all samples at once (vectorized)
    total_diff = np.zeros(N)
    
    for (start, end) in bins:
        # Compute overlap (sum of elementwise minima) for this bin
        overlap = np.sum(np.minimum(vec_i[start:end], totalMatrix[:, start:end]), axis=1)
        
        # Histogram intersection distance term (same as MATLAB logic)
        total_diff += (1 - overlap) ** 2

    # MATLAB code then does: Dsim(:,i) = -((Dsim(:,i)).^2)
    Dsim[:, i] = -(total_diff ** 2)

Dsim = np.maximum(Dsim, Dsim.T)

# Wrap back into a DataFrame for readability
similarity_df = pd.DataFrame(Dsim, index=df.index, columns=df.index)

# print(similarity_df.head())


In [ ]:
# OLD CODE

In [35]:
import h5py

filename = r'Y:\Members\Mia-Sanjana-Hadent\Processed Data\FOR SANJANA Testing_Mouse1\session_1_out_total1.mat'

with h5py.File(filename, 'r') as file:
    if 'Clusters' in file:
        clusters_group = file['Clusters']
        if 'sim' in clusters_group:
            sim_data = clusters_group['sim'][:]
            sim_matrix_data = pd.DataFrame(sim_data)

        if 'idx' in clusters_group:
            idx_data = clusters_group['idx'][:]
            clusters_data = pd.DataFrame(idx_data).transpose()

In [ ]:
# original sim matrix code

clusters_data = clusters_data.rename(columns={0: 'cluster'})

clusters_data_sorted = clusters_data.sort_values(by='cluster')
clusters_data_sorted['cluster'] = clusters_data_sorted['cluster'].astype( int )

sorted_indices = clusters_data_sorted.index
reordered_matrix = sim_matrix_data.loc[sorted_indices, sorted_indices]

cluster_counts = clusters_data_sorted['cluster'].value_counts().reset_index()
cluster_counts.columns = ['cluster', 'count']
cluster_counts = cluster_counts.sort_values(by='cluster')

clusters = cluster_counts['cluster']
counts = cluster_counts['count'].values

final_matrix = np.zeros((len(clusters), len(clusters)))

# Compute averages for each cluster pair
for i, cluster_i in enumerate(clusters):
    for j, cluster_j in enumerate(clusters):
        # Get the indices for rows and columns in the original matrix for the respective clusters
        rows_i = clusters_data_sorted[clusters_data_sorted['cluster'] == cluster_i].index
        rows_j = clusters_data_sorted[clusters_data_sorted['cluster'] == cluster_j].index
        
        # Compute the average of the submatrix
        submatrix = reordered_matrix.loc[rows_i, rows_j]
        # submatrix = sim_matrix_data.loc[rows_i, rows_j]
        final_matrix[i, j] = submatrix.values.mean()

final_matrix_df = pd.DataFrame(final_matrix, index=clusters, columns=clusters)

In [ ]:
# optimize old code?

clusters = np.sort(clusters_data['cluster'].unique())
labels = clusters_data['cluster'].values
sim = sim_matrix_data.values.astype(np.float32)

# Build cluster membership matrix
K = len(clusters)
N = len(labels)
M = np.zeros((N, K), dtype=np.float32)
for k, c in enumerate(clusters):
    M[labels == c, k] = 1.0

# Normalize cluster counts
cluster_sizes = M.sum(axis=0)

# Compute cluster-cluster similarity matrix
numerator = M.T @ sim @ M
denominator = np.outer(cluster_sizes, cluster_sizes)
final_matrix = numerator / denominator

final_matrix_df = pd.DataFrame(final_matrix, index=clusters, columns=clusters)


In [ ]:
final_matrix_df